# 👥 Demographic Data Analysis
**UCI Adult Census Income — 48,842 individuals**

---

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("adult.csv")
df.columns = [c.strip() for c in df.columns]
for c in df.select_dtypes("object").columns:
    df[c] = df[c].str.strip()
inc_col = [c for c in df.columns if "income" in c.lower() or "salary" in c.lower()][0]
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Income column: {inc_col}")
df.head()

## 2. Summary Statistics

In [ ]:
print("=== DATASET SUMMARY ===")
high_pct = df[inc_col].str.contains(">50K", na=False).mean() * 100
print(f"Total records      : {len(df):,}")
print(f"High earners >50K  : {high_pct:.1f}%")
print(f"Avg age            : {df['age'].mean():.1f}")
print(f"Avg hours/week     : {df['hours-per-week'].mean():.1f}")
print(f"Countries          : {df['native-country'].nunique()}")
print(f"Occupations        : {df['occupation'].nunique()}")
print("\nIncome distribution:")
print(df[inc_col].value_counts())

## 3. Income Distribution

In [ ]:
inc_counts = df[inc_col].value_counts().reset_index()
inc_counts.columns = ["Income","Count"]
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"bar"}]])
fig.add_trace(go.Pie(labels=inc_counts["Income"], values=inc_counts["Count"],
                     hole=0.4, name="Income"), row=1, col=1)

edu_inc = df.groupby(["education", inc_col]).size().reset_index(name="Count")
fig2 = px.bar(edu_inc, x="education", y="Count", color=inc_col,
              barmode="group", title="Income by Education Level")
fig2.update_layout(height=420, xaxis_tickangle=-40)
fig2.show()

fig.update_layout(title="Income Distribution", height=380)
fig.show()

## 4. High Earner % by Occupation

In [ ]:
occ_inc = df.groupby("occupation")[inc_col].apply(
    lambda x: x.str.contains(">50K", na=False).mean() * 100
).reset_index()
occ_inc.columns = ["Occupation","High Earner %"]
occ_inc = occ_inc.sort_values("High Earner %", ascending=True)
fig = px.bar(occ_inc, x="High Earner %", y="Occupation", orientation="h",
             color="High Earner %", color_continuous_scale="Viridis",
             title="High Earner % (>50K) by Occupation")
fig.update_layout(height=480)
fig.show()

## 5. Race & Gender Analysis

In [ ]:
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"pie"}]])
race_c = df["race"].value_counts()
sex_c  = df["sex"].value_counts()
fig.add_trace(go.Pie(labels=race_c.index, values=race_c.values, hole=0.4, name="Race"), row=1, col=1)
fig.add_trace(go.Pie(labels=sex_c.index,  values=sex_c.values,  hole=0.4, name="Gender"), row=1, col=2)
fig.update_layout(title="Race and Gender Distribution", height=420)
fig.show()

# High earner % by gender
for g in df["sex"].unique():
    pct = df[df["sex"]==g][inc_col].str.contains(">50K", na=False).mean()*100
    print(f"{g}: {pct:.1f}% high earners")

## 6. Age Distribution by Income

In [ ]:
fig = px.histogram(df, x="age", color=inc_col, nbins=30,
                   barmode="overlay", opacity=0.7,
                   title="Age Distribution by Income Level")
fig.update_layout(height=420)
fig.show()

## 7. Hours/Week by Income

In [ ]:
fig = px.box(df, x=inc_col, y="hours-per-week", color=inc_col,
             title="Hours per Week Distribution by Income",
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=420)
fig.show()

## 8. Top Countries — High Earner %

In [ ]:
country_high = df.groupby("native-country")[inc_col].apply(
    lambda x: x.str.contains(">50K", na=False).mean()*100
).reset_index()
country_high.columns = ["Country","High Earner %"]
country_high = country_high[country_high["Country"] != "?"]
top20_c = country_high.nlargest(20,"High Earner %")
fig = px.bar(top20_c, x="Country", y="High Earner %",
             color="High Earner %", color_continuous_scale="Viridis",
             title="Top 20 Countries — High Earner %")
fig.update_layout(height=420, xaxis_tickangle=-40)
fig.show()

## 9. Correlation Matrix

In [ ]:
num_df = df.select_dtypes(include="number")
corr = num_df.corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                text_auto=".2f", title="Correlation Matrix", aspect="auto")
fig.update_layout(height=420)
fig.show()

## 10. Age vs Hours/Week Scatter

In [ ]:
fig = px.scatter(df.sample(3000), x="age", y="hours-per-week",
                color=inc_col, facet_col="sex", opacity=0.5,
                title="Age vs Hours/Week by Income & Gender")
fig.update_layout(height=450)
fig.show()

## 11. Key Insights

In [ ]:
print("=== DEMOGRAPHICS KEY INSIGHTS ===")
high_edu = ["Bachelors","Masters","Doctorate","Prof-school"]
h_pct = df[df["education"].isin(high_edu)][inc_col].str.contains(">50K", na=False).mean()*100
l_pct = df[~df["education"].isin(high_edu)][inc_col].str.contains(">50K", na=False).mean()*100
print(f"High education earners >50K  : {h_pct:.1f}%")
print(f"Without higher edu >50K      : {l_pct:.1f}%")
print(f"Education premium            : +{h_pct-l_pct:.1f}pp")
print(f"\nTop occupation by high earner %: {occ_inc.nlargest(1,'High Earner %')['Occupation'].values[0]}")
print(f"Avg age (>50K)               : {df[df[inc_col].str.contains('>50K',na=False)]['age'].mean():.1f}")
print(f"Avg age (<=50K)              : {df[~df[inc_col].str.contains('>50K',na=False)]['age'].mean():.1f}")